# F2-M06b: Análisis Temporal — Visualizaciones Altair

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Email** | mjmorteruiz@uoc.edu (UOC) \| morte@uji.es (UJI) |
| **Fase** | Fase 2 — EDA Datos Originales |
| **Módulo** | M06b — Análisis Temporal (Altair) |

---

## 🎯 Qué hace
Complementa el análisis de evolución temporal (M06) con tres visualizaciones
construidas con **Altair/Vega-Lite**, una librería declarativa de visualización
diferente a Plotly. Demuestra versatilidad en herramientas y aporta perspectivas
no cubiertas en M06:
- **Bump chart**: ranking dinámico de ramas por matrículas año a año
- **Heatmap de egreso**: tasa de egreso por rama y curso con valores embebidos
- **Strip plot por cohorte**: distribución de notas según año de entrada

## 📋 Requisitos
- `df_alumno.parquet` en `data/02_processed/`
- Librerías: `altair>=5.0`, `pandas`, `numpy`
- Módulos: `src.config`, `src.utils`, `src.html`

## 📤 Genera
| Archivo | Contenido |
|---|---|
| `docs/html/fase2/m06_temporal.html` | Página HTML con los 3 gráficos Altair |

*(Los gráficos se embeben inline en el HTML — no generan ficheros .html separados)*

## 🔄 Flujo
```
... → M06 Evolución → **M06b Temporal** → M07 Conclusiones → ...
```

## ➡️ Siguiente
`f2_m07_conclusiones.ipynb`


In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN DEL ENTORNO
# ============================================================================
# - Localiza ROOT buscando src/ (robusto, sin hardcodes)
# - Importa módulos del proyecto y Altair
# - Crea directorios de salida
# ============================================================================

import sys
import warnings
import json
from pathlib import Path

warnings.filterwarnings('ignore')

# --- Detectar entorno y localizar ROOT ---
def _encontrar_root(start: Path) -> Path:
    for parent in [start] + list(start.parents):
        if (parent / 'src').is_dir():
            return parent
    raise FileNotFoundError(
        f"No se encontró carpeta 'src/' subiendo desde {start}. "
        f"Verifica que el notebook está dentro de AU_UJI/"
    )

ROOT = _encontrar_root(Path.cwd())
sys.path.insert(0, str(ROOT))

# --- Imports ---
import pandas as pd
import numpy as np
import altair as alt

from src.config import RUTA_PROCESSED, RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.html import (
    generar_kpis_html,
    generar_seccion_html,
    generar_html_navegacion_completa,
    guardar_html
)
from src.html.render import render_pagina_desde_fichero

# --- Rutas de salida ---
RUTA_FASE2 = RUTA_HTML / 'fase2'
crear_directorios([RUTA_FASE2])

info_entorno()
print(f'\n✅ Altair version: {alt.__version__}')


✓ Directorios verificados: 1
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [2]:
# ============================================================================
# CELDA 2: CARGAR DATOS Y CONFIGURACIÓN
# ============================================================================
# Carga df_alumno.parquet y define mapas de nombres de rama y colores.
# Los nombres completos sustituyen las abreviaturas en todos los gráficos.
# ============================================================================

print('=' * 60)
print('CARGANDO DATOS')
print('=' * 60)

df = pd.read_parquet(RUTA_PROCESSED / 'df_alumno.parquet')

# Identificar columna de curso
curso_col = None
for col in ['curso_aca', 'curso_aca_id', 'curso_academico']:
    if col in df.columns:
        curso_col = col
        break
if curso_col is None:
    raise ValueError('No se encontró columna de curso académico')

# Nombres completos de rama (estándar del proyecto)
RAMAS_NOMBRES = {
    'EX': 'Ciencias Experimentales',
    'HU': 'Artes y Humanidades',
    'SA': 'Ciencias de la Salud',
    'SO': 'Ciencias Sociales y Jurídicas',
    'TE': 'Ingeniería y Arquitectura'
}

# Paleta de colores por rama (Dark24 provisional)
COLORES_RAMAS = ['#2E91E5', '#1CA71C', '#E15F99', '#FB0D0D', '#DA16FF']

# Métricas globales para KPIs
cursos = sorted(df[curso_col].unique())
n_cursos = len(cursos)
curso_min, curso_max = cursos[0], cursos[-1]

print(f'✅ Dataset: {len(df):,} filas × {len(df.columns)} columnas')
print(f'📅 Cursos: {n_cursos} ({curso_min} - {curso_max})')
print(f'🌿 Ramas: {sorted(df["rama"].unique())}')


CARGANDO DATOS


✅ Dataset: 109,568 filas × 37 columnas
📅 Cursos: 11 (2010 - 2020)
🌿 Ramas: ['EX', 'HU', 'SA', 'SO', 'TE']


In [3]:
# ============================================================================
# CELDA 3: FUNCIÓN AUXILIAR — ALTAIR CON CDN
# ============================================================================
# Los gráficos Altair se embeben inline en el HTML usando Vega/Vega-Lite
# desde CDN (jsdelivr). Equivalente a include_plotlyjs='cdn' en Plotly.
# Peso por gráfico: ~10-80KB frente a ~3MB embebido.
# ============================================================================

# Flag para incluir los scripts CDN solo una vez en la página
_cdn_incluido = False

def altair_a_html(chart, div_id: str, incluir_cdn: bool = False) -> str:
    """
    Convierte un gráfico Altair a HTML inline con Vega desde CDN.

    Parameters
    ----------
    chart : alt.Chart
        Gráfico Altair a convertir.
    div_id : str
        ID único del div contenedor.
    incluir_cdn : bool
        Si True, añade los scripts CDN de Vega/Vega-Lite/Vega-Embed.
        Solo necesario una vez por página.
    """
    spec_json = json.dumps(chart.to_dict())
    cdn = ''
    if incluir_cdn:
        cdn = (
            '<script src="https://cdn.jsdelivr.net/npm/vega@5"></script>\n'
            '<script src="https://cdn.jsdelivr.net/npm/vega-lite@5"></script>\n'
            '<script src="https://cdn.jsdelivr.net/npm/vega-embed@6"></script>\n'
        )
    return (
        f'{cdn}'
        f'<div id="{div_id}" style="width:100%; overflow-x:auto"></div>\n'
        f'<script>vegaEmbed("#{div_id}", {spec_json}, '
        f'{{renderer:"svg", actions:false}});</script>\n'
    )

print('✅ Helper altair_a_html() definido')


✅ Helper altair_a_html() definido


In [4]:
# ============================================================================
# CELDA 4: GRÁFICO 1 — BUMP CHART RANKING DE RAMAS
# ============================================================================
# Muestra cómo cambia el ranking de cada rama por número de matrículas
# a lo largo de los 11 cursos académicos.
# Tipo de gráfico exclusivo de Altair — Plotly no tiene bump chart nativo.
# ============================================================================

print('=' * 60)
print('GRÁFICO 1: BUMP CHART — RANKING DE RAMAS')
print('=' * 60)

# Preparar datos: matrículas por curso y rama → ranking
rama_curso = df.groupby([curso_col, 'rama']).size().reset_index(name='matriculas')
rama_curso['ranking'] = (
    rama_curso.groupby(curso_col)['matriculas']
    .rank(ascending=False).astype(int)
)
rama_curso['curso_str'] = rama_curso[curso_col].astype(str)
rama_curso['rama_nombre'] = rama_curso['rama'].map(RAMAS_NOMBRES).fillna(rama_curso['rama'])

ramas_unicas = sorted(rama_curso['rama_nombre'].unique())

# Líneas del bump chart
lineas = alt.Chart(rama_curso).mark_line(strokeWidth=3, point=True).encode(
    x=alt.X('curso_str:O', title='Curso Académico',
            axis=alt.Axis(labelAngle=-45)),
    y=alt.Y('ranking:Q', title='Ranking (1 = más matrículas)',
            scale=alt.Scale(domain=[5.5, 0.5]),
            axis=alt.Axis(tickCount=5)),
    color=alt.Color('rama_nombre:N', title='Rama',
                    scale=alt.Scale(domain=ramas_unicas, range=COLORES_RAMAS)),
    tooltip=['curso_str:O', 'rama_nombre:N', 'ranking:Q', 'matriculas:Q']
)

# Etiquetas en el primer año
primer_curso = rama_curso[rama_curso[curso_col] == rama_curso[curso_col].min()]
etiquetas = alt.Chart(primer_curso).mark_text(
    align='right', dx=-8, fontSize=10
).encode(
    x='curso_str:O', y='ranking:Q', text='rama_nombre:N',
    color=alt.Color('rama_nombre:N',
                    scale=alt.Scale(domain=ramas_unicas, range=COLORES_RAMAS),
                    legend=None)
)

bump_chart = (lineas + etiquetas).properties(
    width=580, height=320,
    title=alt.TitleParams(
        'Ranking de Ramas por Matrículas (Bump Chart)',
        subtitle='Cada línea representa una rama — posición 1 = más matrículas ese curso',
        fontSize=14, subtitleFontSize=11, subtitleColor='#718096'
    )
)

# Convertir a HTML inline con CDN (primera vez → incluir scripts)
bump_html = altair_a_html(bump_chart, 'm06t_bump', incluir_cdn=True)
print('✅ Bump chart generado')
print(f'   Ramas: {ramas_unicas}')


GRÁFICO 1: BUMP CHART — RANKING DE RAMAS
✅ Bump chart generado
   Ramas: ['Artes y Humanidades', 'Ciencias Experimentales', 'Ciencias Sociales y Jurídicas', 'Ciencias de la Salud', 'Ingeniería y Arquitectura']


In [5]:
# ============================================================================
# CELDA 5: GRÁFICO 2 — HEATMAP TASA DE EGRESO POR RAMA Y CURSO
# ============================================================================
# Muestra qué porcentaje de alumnos de cada rama egresó cada año.
# Los valores se embeben directamente en las celdas del heatmap.
# Perspectiva diferente al heatmap de matrículas de M06.
# ============================================================================

print('=' * 60)
print('GRÁFICO 2: HEATMAP TASA DE EGRESO')
print('=' * 60)

# Tasa de egreso = egresados / total matrículas ese año y rama
egreso = (
    df.groupby([curso_col, 'rama'])
    .apply(lambda x: (x['egresado'] == 'S').sum() / len(x) * 100)
    .reset_index(name='tasa_egreso')
)
egreso['curso_str'] = egreso[curso_col].astype(str)
egreso['rama_nombre'] = egreso['rama'].map(RAMAS_NOMBRES).fillna(egreso['rama'])

# Rectángulos coloreados
rect = alt.Chart(egreso).mark_rect(stroke='white', strokeWidth=1).encode(
    x=alt.X('curso_str:O', title='Curso Académico',
            axis=alt.Axis(labelAngle=-45)),
    y=alt.Y('rama_nombre:N', title=None),
    color=alt.Color('tasa_egreso:Q', title='Tasa Egreso (%)',
                    scale=alt.Scale(scheme='blues')),
    tooltip=['curso_str:O', 'rama_nombre:N',
             alt.Tooltip('tasa_egreso:Q', format='.1f', title='Tasa Egreso (%)')]
)

# Valores numéricos embebidos en cada celda
texto = alt.Chart(egreso).mark_text(fontSize=10, fontWeight='bold').encode(
    x='curso_str:O', y='rama_nombre:N',
    text=alt.Text('tasa_egreso:Q', format='.0f'),
    color=alt.condition(
        alt.datum.tasa_egreso > 15,
        alt.value('white'), alt.value('#2d3748')
    )
)

heatmap_chart = (rect + texto).properties(
    width=580, height=200,
    title=alt.TitleParams(
        'Tasa de Egreso (%) por Rama y Curso Académico',
        subtitle='% de alumnos matriculados ese año que constan como egresados',
        fontSize=14, subtitleFontSize=11, subtitleColor='#718096'
    )
)

heatmap_html = altair_a_html(heatmap_chart, 'm06t_heatmap', incluir_cdn=False)
print('✅ Heatmap de egreso generado')


GRÁFICO 2: HEATMAP TASA DE EGRESO


✅ Heatmap de egreso generado


In [6]:
# ============================================================================
# CELDA 6: GRÁFICO 3 — STRIP PLOT DE NOTAS POR COHORTE DE ENTRADA
# ============================================================================
# Perspectiva diferente a M06: agrupa por AÑO DE ENTRADA del alumno
# (no por año académico). Muestra si las cohortes más recientes tienen
# mejor o peor rendimiento. Incluye puntos individuales + media ± std.
# ============================================================================

print('=' * 60)
print('GRÁFICO 3: STRIP PLOT POR COHORTE DE ENTRADA')
print('=' * 60)

# Filtrar registros con nota y año de entrada disponibles
df_notas = df[df['media_curso'].notna() & df['curso_aca_ini'].notna()].copy()
df_notas['cohorte'] = df_notas['curso_aca_ini'].astype(str)

# Muestra aleatoria por cohorte (máx 150) para rendimiento visual
partes = []
for coh, grupo in df_notas.groupby('cohorte'):
    partes.append(
        grupo[['cohorte', 'media_curso']].sample(
            min(150, len(grupo)), random_state=42
        )
    )
muestra_df = pd.concat(partes, ignore_index=True)

# Media y desviación estándar por cohorte
medias_df = (
    df_notas.groupby('cohorte')['media_curso']
    .agg(media='mean', std='std')
    .reset_index()
)
medias_df['ymin'] = medias_df['media'] - medias_df['std']
medias_df['ymax'] = medias_df['media'] + medias_df['std']

# Puntos individuales (semitransparentes)
puntos = alt.Chart(muestra_df).mark_circle(size=18, opacity=0.25).encode(
    x=alt.X('cohorte:O', title='Cohorte (año de entrada)',
            axis=alt.Axis(labelAngle=-45)),
    y=alt.Y('media_curso:Q', title='Nota media del curso',
            scale=alt.Scale(domain=[0, 10])),
    color=alt.Color('cohorte:O', legend=None,
                    scale=alt.Scale(scheme='tableau10')),
    tooltip=['cohorte:O', alt.Tooltip('media_curso:Q', format='.2f', title='Nota')]
)

# Barra de desviación estándar
barra_std = alt.Chart(medias_df).mark_rule(
    color='#2d3748', strokeWidth=2
).encode(
    x='cohorte:O',
    y=alt.Y('ymin:Q', title=''),
    y2='ymax:Q'
)

# Punto de media
punto_media = alt.Chart(medias_df).mark_point(
    size=100, filled=True, color='#2d3748'
).encode(
    x='cohorte:O', y='media:Q',
    tooltip=[
        'cohorte:O',
        alt.Tooltip('media:Q', format='.2f', title='Media'),
        alt.Tooltip('std:Q', format='.2f', title='Desv. Std')
    ]
)

strip_chart = (puntos + barra_std + punto_media).properties(
    width=580, height=300,
    title=alt.TitleParams(
        'Distribución de Notas por Cohorte de Entrada',
        subtitle='Puntos = muestra aleatoria por cohorte | ━ Media ± desv. estándar',
        fontSize=14, subtitleFontSize=11, subtitleColor='#718096'
    )
)

strip_html = altair_a_html(strip_chart, 'm06t_strip', incluir_cdn=False)
n_cohortes = df_notas['cohorte'].nunique()
print(f'✅ Strip plot generado — {n_cohortes} cohortes, {len(muestra_df):,} puntos')


GRÁFICO 3: STRIP PLOT POR COHORTE DE ENTRADA


✅ Strip plot generado — 12 cohortes, 1,800 puntos


In [7]:
# ============================================================================
# CELDA 7: GENERAR HTML
# ============================================================================
# Los gráficos Altair se embeben INLINE (sin iframe, sin ficheros externos).
# Los scripts CDN de Vega se incluyen una sola vez (en bump_html).
# ============================================================================

print('=' * 60)
print('GENERANDO HTML')
print('=' * 60)

# --- Generar HTML ---
html_completo = render_pagina_desde_fichero(
    'f2_m06_temporal.ipynb',
    bump_html,
    carpeta_notebook='fase2_eda'
)
ruta_html = RUTA_FASE2 / 'm06_temporal.html'
ruta_html.parent.mkdir(parents=True, exist_ok=True)
ruta_html.write_text(html_completo, encoding='utf-8')
print(f'✅ HTML generado: {ruta_html}')
print(f'✅ HTML generado: {ruta_html}')


GENERANDO HTML
✅ HTML generado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase2\m06_temporal.html
✅ HTML generado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase2\m06_temporal.html


In [8]:
# ============================================================================
# CELDA 8: RESUMEN
# ============================================================================

print('\n' + '=' * 60)
print('✅ F2-M06b TEMPORAL COMPLETADO')
print('=' * 60)
print('📈 Bump chart: ranking de ramas por matrículas')
print('🔥 Heatmap: tasa de egreso por rama y curso')
print('🎯 Strip plot: notas por cohorte de entrada')
print(f'\n📁 HTML: {ruta_html}')
print('\n📌 Siguiente: f2_m07_conclusiones.ipynb')
print('=' * 60)



✅ F2-M06b TEMPORAL COMPLETADO
📈 Bump chart: ranking de ramas por matrículas
🔥 Heatmap: tasa de egreso por rama y curso
🎯 Strip plot: notas por cohorte de entrada

📁 HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase2\m06_temporal.html

📌 Siguiente: f2_m07_conclusiones.ipynb
